In [1]:
import os
import sys
import torch
import numpy as np
import networkx as nx
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Project imports
from equinox.config import get_config
from equinox.sampling.trespass.sampler_log import sample_tres_trajectory
from equinox.sampling.sample_route import sample_routes
from equinox.dp.trespass.sparse_io_utils import load_sparse_coo_tensor_with_convention

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Imports successful!")


ImportError: cannot import name 'get_config' from 'equinox.config' (D:\project-equinox\src\equinox\config.py)

In [ ]:
# Load configuration
config = get_config('../default.yaml')
print(f"Loaded configuration for route: {config.origin_airport_id} -> {config.destination_airport_id}")
print(f"Date: {config.date}")

# Define data paths
data_dir = Path('../data')
graph_dir = data_dir / 'graph'
results_dir = data_dir / 'results'
trajectories_dir = graph_dir / 'trajectories'

# Create trajectories directory if it doesn't exist
trajectories_dir.mkdir(exist_ok=True, parents=True)

print(f"Data directory: {data_dir}")
print(f"Graph directory: {graph_dir}")
print(f"Trajectories directory: {trajectories_dir}")


In [ ]:
# Load route graph
graph_file = graph_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}.gml"
if not graph_file.exists():
    print(f"Graph file not found: {graph_file}")
    print("Available graph files:")
    for f in graph_dir.glob("*.gml"):
        print(f"  {f.name}")
    raise FileNotFoundError(f"Graph file not found: {graph_file}")

print(f"Loading graph from: {graph_file}")
G = nx.read_gml(str(graph_file))

# Create node mappings
node_list = list(G.nodes())
node_to_idx = {node_id: i for i, node_id in enumerate(node_list)}
idx_to_node = {i: node_id for node_id, i in node_to_idx.items()}

print(f"Graph loaded: {len(G.nodes())} nodes, {len(G.edges())} edges")
print(f"Origin: {config.origin_airport_id} (idx: {node_to_idx.get(config.origin_airport_id, 'NOT FOUND')})")
print(f"Destination: {config.destination_airport_id} (idx: {node_to_idx.get(config.destination_airport_id, 'NOT FOUND')})")


In [ ]:
# Load backward value functions and edge costs for TResPASS sampling
try:
    # Try to load the soft cost-to-go (V_bwd)
    v_soft_dir = graph_dir / 'V_soft'
    v_bwd_file = v_soft_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_V_BWD_SPRSE_WIND.pt"
    
    if v_bwd_file.exists():
        print(f"Loading V_bwd from: {v_bwd_file}")
        V_bwd_sparse = torch.load(v_bwd_file, map_location='cpu')
        print(f"V_bwd sparse tensor loaded: {V_bwd_sparse.shape if hasattr(V_bwd_sparse, 'shape') else 'sparse tensor'}")
        
        # Convert sparse to dense if needed
        if hasattr(V_bwd_sparse, 'to_dense'):
            V_bwd_dense = V_bwd_sparse.to_dense()
        else:
            V_bwd_dense = V_bwd_sparse
        
        print(f"V_bwd dense shape: {V_bwd_dense.shape}")
        print(f"V_bwd value range: [{torch.min(V_bwd_dense):.3f}, {torch.max(V_bwd_dense):.3f}]")
        
        trespass_data_available = True
    else:
        print(f"V_bwd file not found: {v_bwd_file}")
        print("Available V_soft files:")
        for f in v_soft_dir.glob("*.pt"):
            print(f"  {f.name}")
        trespass_data_available = False
        
except Exception as e:
    print(f"Error loading TResPASS data: {e}")
    trespass_data_available = False


In [ ]:
# Load edge costs for TResPASS sampling
if trespass_data_available:
    try:
        transitions_dir = graph_dir / 'transitions'
        edge_costs_file = transitions_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_CLB_WIND.pkl"
        
        if edge_costs_file.exists():
            print(f"Loading edge costs from: {edge_costs_file}")
            edge_costs_tensor = load_sparse_coo_tensor_with_convention(str(edge_costs_file))
            print(f"Edge costs tensor loaded: indices shape {edge_costs_tensor.indices().shape}, values shape {edge_costs_tensor.values().shape}")
            print(f"Edge costs value range: [{torch.min(edge_costs_tensor.values()):.3f}, {torch.max(edge_costs_tensor.values()):.3f}]")
        else:
            print(f"Edge costs file not found: {edge_costs_file}")
            print("Available transition files:")
            for f in transitions_dir.glob("*.pkl"):
                print(f"  {f.name}")
            trespass_data_available = False
            
    except Exception as e:
        print(f"Error loading edge costs: {e}")
        trespass_data_available = False
        
print(f"TResPASS data available: {trespass_data_available}")


In [ ]:
# TResPASS trajectory sampling
if trespass_data_available:
    print("=== TResPASS Trajectory Sampling ===")
    
    # Sampling parameters
    origin_node_id = config.origin_airport_id
    goal_node_id = config.destination_airport_id
    initial_rho = 36  # Initial climb time remaining bin index
    initial_phase = 0  # 0: CLIMB, 1: CRUISE, 2: DESCENT
    gamma = 0.1  # Temperature parameter
    num_samples = 100
    max_steps_per_trajectory = 200
    
    print(f"Sampling {num_samples} trajectories from {origin_node_id} to {goal_node_id}")
    print(f"Initial state: rho={initial_rho}, phase={initial_phase}, gamma={gamma}")
    
    # Sample trajectories
    successful_trajectories = []
    trajectory_costs_list = []
    failed_samples = 0
    
    for i in range(num_samples):
        try:
            trajectory, trajectory_costs = sample_tres_trajectory(
                G=G,
                node_to_idx=node_to_idx,
                idx_to_node=idx_to_node,
                origin_node_id=origin_node_id,
                goal_node_id=goal_node_id,
                initial_rho=initial_rho,
                initial_phase=initial_phase,
                soft_cost_to_go=V_bwd_dense,
                edge_costs_uv=edge_costs_tensor,
                gamma=gamma,
                max_steps=max_steps_per_trajectory
            )
            
            if trajectory and len(trajectory) > 1:
                successful_trajectories.append(trajectory)
                trajectory_costs_list.append(trajectory_costs)
                
                if (i + 1) % 20 == 0:
                    print(f"Sampled {i + 1}/{num_samples} trajectories...")
            else:
                failed_samples += 1
                
        except Exception as e:
            failed_samples += 1
            if i < 5:  # Only print first few errors
                print(f"Sampling failed for trajectory {i + 1}: {str(e)[:100]}...")
    
    print(f"\nTResPASS Sampling Results:")
    print(f"Successful trajectories: {len(successful_trajectories)}")
    print(f"Failed samples: {failed_samples}")
    print(f"Success rate: {len(successful_trajectories) / num_samples * 100:.1f}%")
    
    if successful_trajectories:
        trajectory_lengths = [len(t) for t in successful_trajectories]
        total_costs = [sum(costs) for costs in trajectory_costs_list]
        
        print(f"Average trajectory length: {np.mean(trajectory_lengths):.2f} ± {np.std(trajectory_lengths):.2f}")
        print(f"Average total cost: {np.mean(total_costs):.3f} ± {np.std(total_costs):.3f}")
        print(f"Length range: [{min(trajectory_lengths)}, {max(trajectory_lengths)}]")
        print(f"Cost range: [{min(total_costs):.3f}, {max(total_costs):.3f}]")
else:
    print("TResPASS data not available, skipping TResPASS sampling")
    successful_trajectories = []


In [ ]:
# Save TResPASS trajectories to file
if successful_trajectories:
    trajectory_file = trajectories_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_trespass_trajectories.txt"
    
    with open(trajectory_file, 'w') as f:
        for i, (trajectory, costs) in enumerate(zip(successful_trajectories, trajectory_costs_list)):
            # Extract waypoint names from trajectory tuples (waypoint_name, k, rho, phase)
            waypoint_names = [str(step[0]) for step in trajectory]
            trajectory_line = " ".join(waypoint_names)
            total_cost = sum(costs)
            
            # Save as "cost,trajectory"
            f.write(f"{total_cost:.6f},{trajectory_line}\n")
    
    print(f"Saved {len(successful_trajectories)} TResPASS trajectories to: {trajectory_file}")
    
    # Show sample trajectories
    print("\nSample trajectories:")
    for i, (trajectory, costs) in enumerate(zip(successful_trajectories[:3], trajectory_costs_list[:3])):
        waypoints = [step[0] for step in trajectory]
        total_cost = sum(costs)
        print(f"{i+1}. Cost: {total_cost:.3f}, Length: {len(waypoints)}, Route: {' -> '.join(waypoints[:3])} ... {' -> '.join(waypoints[-2:])}")


In [ ]:
def analyze_trajectory_occurrences(trajectory_file_path: str, return_type='probability'):
    """
    Analyze the occurrence (count or probability) and costs of each route.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file. Each line should be `cost,route`.
        return_type (str): Either 'count' for raw counts or 'probability' for normalized probabilities
        
    Returns:
        A tuple of (results, costs):
        - results (dict): Dictionary mapping trajectory strings to their occurrence counts or probabilities.
        - costs (dict): Dictionary mapping trajectory strings to a list of their costs.
    """
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}, {}
    
    routes = []
    costs = defaultdict(list)
    
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            parts = line.split(',', 1)
            if len(parts) == 2:
                cost_str, route = parts
                routes.append(route)
                try:
                    costs[route].append(float(cost_str))
                except ValueError:
                    print(f"Warning: Could not parse cost '{cost_str}' for route. Skipping cost for this entry.")
            else:
                # Handle lines without a cost for backward compatibility
                routes.append(line)

    if not routes:
        return {}, {}
    
    route_counts = Counter(routes)
    
    if return_type == 'count':
        return dict(route_counts), dict(costs)
    elif return_type == 'probability':
        total_routes = len(routes)
        route_probabilities = {
            route: count / total_routes 
            for route, count in route_counts.items()
        }
        return route_probabilities, dict(costs)
    else:
        raise ValueError("return_type must be either 'count' or 'probability'")

def get_trajectory_statistics(trajectory_file_path: str):
    """
    Get comprehensive statistics about trajectories in a file.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        
    Returns:
        dict: Dictionary containing various statistics
    """
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}
    
    trajectory_lengths = []
    trajectory_costs = []
    unique_routes = set()
    
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(',', 1)
            if len(parts) == 2:
                cost_str, route_str = parts
                try:
                    trajectory_costs.append(float(cost_str))
                except ValueError:
                    print(f"Warning: Could not parse cost '{cost_str}'.")
            else:
                route_str = line
            
            waypoints = route_str.split()
            trajectory_lengths.append(len(waypoints))
            unique_routes.add(route_str)
    
    if not trajectory_lengths:
        return {}
    
    stats = {
        'total_trajectories': len(trajectory_lengths),
        'unique_trajectories': len(unique_routes),
        'diversity_ratio': len(unique_routes) / len(trajectory_lengths) if trajectory_lengths else 0,
        'avg_length': np.mean(trajectory_lengths),
        'median_length': np.median(trajectory_lengths),
        'min_length': np.min(trajectory_lengths),
        'max_length': np.max(trajectory_lengths),
        'std_length': np.std(trajectory_lengths)
    }
    
    if trajectory_costs:
        stats.update({
            'avg_cost': np.mean(trajectory_costs),
            'median_cost': np.median(trajectory_costs),
            'min_cost': np.min(trajectory_costs),
            'max_cost': np.max(trajectory_costs),
            'std_cost': np.std(trajectory_costs)
        })
    
    return stats

print("Trajectory analysis functions defined!")


In [ ]:
def print_trajectory_analysis(trajectory_file_path: str, top_n: int = 10):
    """
    Print comprehensive trajectory analysis.
    """
    # Get statistics
    stats = get_trajectory_statistics(trajectory_file_path)
    if not stats:
        return
    
    print("=== TRAJECTORY STATISTICS ===")
    print(f"Total trajectories:     {stats['total_trajectories']}")
    print(f"Unique trajectories:    {stats['unique_trajectories']}")
    print(f"Diversity ratio:        {stats['diversity_ratio']:.3f}")
    print()
    print("Trajectory Length Statistics:")
    print(f"Average length:         {stats['avg_length']:.2f}")
    print(f"Median length:          {stats['median_length']:.1f}")
    print(f"Min length:             {stats['min_length']}")
    print(f"Max length:             {stats['max_length']}")
    print(f"Standard deviation:     {stats['std_length']:.2f}")

    if 'avg_cost' in stats:
        print()
        print("Trajectory Cost Statistics:")
        print(f"Average cost:           {stats['avg_cost']:.2f}")
        print(f"Median cost:            {stats['median_cost']:.2f}")
        print(f"Min cost:               {stats['min_cost']:.2f}")
        print(f"Max cost:               {stats['max_cost']:.2f}")
        print(f"Standard deviation:     {stats['std_cost']:.2f}")
    
    # Get occurrence analysis
    results, costs = analyze_trajectory_occurrences(trajectory_file_path, 'probability')
    
    if results:
        print()
        print(f"=== TOP {top_n} MOST FREQUENT TRAJECTORIES ===")
        sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
        
        print(f"{'Rank':<5} {'Probability':<12} {'Avg Cost':<10} {'Trajectory'}")
        print("-" * 90)
        
        for i, (trajectory, prob) in enumerate(sorted_results[:top_n], 1):
            avg_cost = np.mean(costs[trajectory]) if trajectory in costs and costs[trajectory] else 0.0
            
            # Truncate long trajectories for display
            display_trajectory = trajectory
            if len(trajectory) > 60:
                waypoints = trajectory.split()
                if len(waypoints) > 8:
                    display_trajectory = " ".join(waypoints[:4]) + " ... " + " ".join(waypoints[-4:])
            
            print(f"{i:<5} {prob:<12.4f} {avg_cost:<10.3f} {display_trajectory}")

# Analyze trajectories if available
if successful_trajectories:
    trajectory_file = trajectories_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_trespass_trajectories.txt"
    print_trajectory_analysis(str(trajectory_file), top_n=15)
else:
    print("No trajectories available for analysis")


In [ ]:
# Visualize trajectory statistics
if successful_trajectories:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Length distribution
    trajectory_lengths = [len(t) for t in successful_trajectories]
    axes[0, 0].hist(trajectory_lengths, bins=20, alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Trajectory Length (# waypoints)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Trajectory Lengths')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Cost distribution
    if trajectory_costs_list:
        total_costs = [sum(costs) for costs in trajectory_costs_list]
        axes[0, 1].hist(total_costs, bins=20, alpha=0.7, edgecolor='black', color='orange')
        axes[0, 1].set_xlabel('Total Trajectory Cost')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Distribution of Trajectory Costs')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Length vs Cost scatter
    if trajectory_costs_list:
        axes[1, 0].scatter(trajectory_lengths, total_costs, alpha=0.6)
        axes[1, 0].set_xlabel('Trajectory Length')
        axes[1, 0].set_ylabel('Total Cost')
        axes[1, 0].set_title('Length vs Cost Relationship')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Waypoint frequency (top 15)
    waypoint_counts = Counter()
    for trajectory in successful_trajectories:
        for step in trajectory:
            waypoint = step[0]  # Extract waypoint name from (waypoint, k, rho, phase)
            if waypoint not in [config.origin_airport_id, config.destination_airport_id]:
                waypoint_counts[waypoint] += 1
    
    if waypoint_counts:
        top_waypoints = waypoint_counts.most_common(15)
        waypoints, counts = zip(*top_waypoints)
        
        axes[1, 1].barh(range(len(waypoints)), counts)
        axes[1, 1].set_yticks(range(len(waypoints)))
        axes[1, 1].set_yticklabels(waypoints)
        axes[1, 1].set_xlabel('Frequency')
        axes[1, 1].set_title('Top 15 Most Used Waypoints')
        axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nVisualization complete for {len(successful_trajectories)} trajectories")
else:
    print("No trajectories available for visualization")


In [ ]:
# Create a summary report
summary_file = trajectories_dir / f"{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_sampling_summary.txt"

with open(summary_file, 'w') as f:
    f.write(f"Trajectory Sampling Summary\n")
    f.write(f"{'=' * 50}\n\n")
    f.write(f"Route: {config.origin_airport_id} -> {config.destination_airport_id}\n")
    f.write(f"Date: {config.date}\n")
    f.write(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    if successful_trajectories:
        f.write(f"TResPASS Sampling Results:\n")
        f.write(f"- Successful trajectories: {len(successful_trajectories)}\n")
        f.write(f"- Failed samples: {failed_samples}\n")
        f.write(f"- Success rate: {len(successful_trajectories) / num_samples * 100:.1f}%\n")
        
        trajectory_lengths = [len(t) for t in successful_trajectories]
        if trajectory_costs_list:
            total_costs = [sum(costs) for costs in trajectory_costs_list]
            f.write(f"- Average trajectory length: {np.mean(trajectory_lengths):.2f} ± {np.std(trajectory_lengths):.2f}\n")
            f.write(f"- Average total cost: {np.mean(total_costs):.3f} ± {np.std(total_costs):.3f}\n")
            f.write(f"- Length range: [{min(trajectory_lengths)}, {max(trajectory_lengths)}]\n")
            f.write(f"- Cost range: [{min(total_costs):.3f}, {max(total_costs):.3f}]\n")
        
        # Get unique trajectories count
        waypoint_trajectories = []
        for trajectory in successful_trajectories:
            waypoints = [step[0] for step in trajectory]
            waypoint_trajectories.append(" ".join(waypoints))
        
        unique_trajectories = len(set(waypoint_trajectories))
        f.write(f"- Unique trajectories: {unique_trajectories}\n")
        f.write(f"- Diversity ratio: {unique_trajectories / len(successful_trajectories):.3f}\n")
    else:
        f.write("No successful trajectories generated.\n")
    
    f.write(f"\nFiles generated:\n")
    if successful_trajectories:
        f.write(f"- Trajectories: {trajectories_dir / f'{config.origin_airport_id}_{config.destination_airport_id}_{config.date}_trespass_trajectories.txt'}\n")
    f.write(f"- Summary: {summary_file}\n")

print(f"Summary report saved to: {summary_file}")
print("\n=== Trajectory Sampling Complete ===")

# Print final summary to console
if successful_trajectories:
    print(f"\n📊 FINAL RESULTS:")
    print(f"✅ Successfully sampled {len(successful_trajectories)} trajectories")
    print(f"❌ Failed samples: {failed_samples}")
    print(f"📈 Success rate: {len(successful_trajectories) / num_samples * 100:.1f}%")
    
    unique_count = len(set([" ".join([step[0] for step in traj]) for traj in successful_trajectories]))
    print(f"🔄 Unique trajectories: {unique_count}")
    print(f"🎯 Diversity ratio: {unique_count / len(successful_trajectories):.3f}")
    
    if trajectory_costs_list:
        avg_cost = np.mean([sum(costs) for costs in trajectory_costs_list])
        print(f"💰 Average cost: {avg_cost:.3f}")
else:
    print("❌ No successful trajectories were generated")
    print("💡 Try checking if the required data files exist and are properly formatted")
